# CUMULUS — Preprocessing Benchmark Notebook

Dieses Notebook reproduziert die Kern-Experimente aus dem Paper **CUMULUS: Cleaning, Unifying, and Modeling Unstable Eye-tracking Signals**.

## Ziel
- Missing-Value-Imputation (MCAR-Injektion, Remove-and-Reconstruct)
- Outlier-Handling (synthetische Outlier-Injektion)
- Normalisierung (Min–Max / Z-Score / Robust)

## Voraussetzungen
- Python ≥ 3.10
- Pakete: numpy, pandas, scipy, scikit-learn, matplotlib

## Daten
- Erwartet CSV-Dateien wie im Artifact (z.B. `data/raw/*.csv`).
- **Keine absoluten Pfade**: Pfade werden unten konfiguriert.

## Output
- Ergebnisse werden unter `results/` abgelegt (CSV).
- Figuren optional unter `figures/`.


In [ ]:
# [Setup] Imports & globale Konfiguration
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, ks_2samp, zscore, median_abs_deviation, skew, kurtosis

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt

# Reproduzierbarkeit
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Projekt-Struktur (relativ zum Notebook)
PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Relevante Features (Paper: gaze + pupil)
FEATURE_COLS = ["Gaze X", "Gaze Y", "ET_PupilLeft", "ET_PupilRight"]

# Injektionslevel in %
MISSING_LEVELS = [5, 10, 15, 20]
OUTLIER_LEVELS = [5, 10, 15, 20]

# Optional: schneller Run nur mit einer Datei
SINGLE_FILE = None  # z.B. "Frage 1_001_Anonymous 09-05-23 12h04m.csv"


## 1) Daten laden
**Was diese Zellen tun:**
- liest eine oder mehrere CSV-Dateien
- filtert auf die im Paper verwendeten Features



In [ ]:
# [Load] Lade alle CSVs oder eine einzelne Datei
def list_csv_files(data_dir: Path, single_file: str | None = None) -> list[Path]:
    if single_file:
        p = data_dir / single_file
        if not p.exists():
            raise FileNotFoundError(f"Datei nicht gefunden: {p}")
        return [p]
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"Keine CSVs in {data_dir}. Erwartet z.B. data/raw/*.csv")
    return files

def load_csv(path: Path, feature_cols: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name}: Spalten fehlen: {missing_cols}")
    return df[feature_cols].copy()

csv_files = list_csv_files(DATA_DIR, SINGLE_FILE)
print(f"Gefundene Dateien: {len(csv_files)}")
print("Beispiel:", csv_files[0].name)

df0 = load_csv(csv_files[0], FEATURE_COLS)
df0.head()


## 2) Korruptionen injizieren (MCAR Missingness + synthetische Outlier)
**Was diese Zellen tun:**
- erzeugen kontrollierte Störungen (wie im Paper)

**Wichtig:** Missingness ist **MCAR** (kontrollierter Benchmark), Outlier werden als starke Ausreißer addiert.


In [ ]:
# [Corruptions] Missingness (MCAR) & Outlier-Injektion

def introduce_missing_values_mcar(df: pd.DataFrame, cols: list[str], missing_percent: float, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in cols:
        k = int(n * missing_percent / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        out.loc[idx, col] = np.nan
    return out

def introduce_outliers_additive(df: pd.DataFrame, cols: list[str], outlier_percent: float, scale_range=(5.0, 10.0), seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Addiert starke Abweichungen proportional zur Spalten-Std.

    Zweck: reproduzierbare, klare Outlier für Benchmarking (nicht physiologische Sakkaden).
    """
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in cols:
        k = int(n * outlier_percent / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        std = np.nanstd(out[col].to_numpy())
        mult = rng.uniform(scale_range[0], scale_range[1], size=k)
        out.loc[idx, col] = out.loc[idx, col] + mult * std
    return out

# Quick sanity check
tmp = introduce_missing_values_mcar(df0, FEATURE_COLS, 10)
tmp.isna().mean()


## 3) Imputation-Benchmark
**Was diese Zellen tun:**
- definieren Imputer (Mean, LOCF, KNN, GPR, MICE)
- evaluieren per RMSE/MAE (remove-and-reconstruct) und Wilcoxon (Distribution)




In [ ]:
# [Imputation] Methoden

def mean_imputation(df: pd.DataFrame) -> pd.DataFrame:
    imp = SimpleImputer(strategy="mean")
    return pd.DataFrame(imp.fit_transform(df), columns=df.columns, index=df.index)

def locf_imputation(df: pd.DataFrame) -> pd.DataFrame:
    # ffill + bfill, damit Leading-NaNs auch gefüllt werden (für Benchmarking)
    return df.ffill().bfill()

def knn_imputation(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    imp = KNNImputer(n_neighbors=k)
    return pd.DataFrame(imp.fit_transform(df), columns=df.columns, index=df.index)

def gpr_imputation_1d(series: pd.Series) -> pd.Series:
    """GPR pro Feature über Zeitindex (teuer; nur als Baseline)."""
    y = series.to_numpy()
    x = np.arange(len(series)).reshape(-1, 1)
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return series.copy()
    kernel = C(1.0, (1e-3, 1e3)) * RBF(10.0, (1e-2, 1e2))
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, random_state=RANDOM_SEED)
    gp.fit(x[mask], y[mask])
    y_pred = y.copy()
    miss = ~mask
    if miss.any():
        y_pred[miss] = gp.predict(x[miss])
    return pd.Series(y_pred, index=series.index, name=series.name)

def gpr_imputation(df: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([gpr_imputation_1d(df[c]) for c in df.columns], axis=1)

def mice_imputation(df: pd.DataFrame, max_iter: int = 10) -> pd.DataFrame:
    imp = IterativeImputer(max_iter=max_iter, random_state=RANDOM_SEED)
    return pd.DataFrame(imp.fit_transform(df), columns=df.columns, index=df.index)

IMPUTERS = {
    "mean": lambda d: mean_imputation(d),
    "locf": lambda d: locf_imputation(d),
    "knn(k=5)": lambda d: knn_imputation(d, k=5),
    "gpr": lambda d: gpr_imputation(d),
    "mice": lambda d: mice_imputation(d, max_iter=10),
}

def remove_and_reconstruct(original: pd.DataFrame, imputer_fn, missing_percent: float, seed: int = RANDOM_SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Maskiert *beobachtete* Werte MCAR und rekonstruiert. Gibt (masked, imputed) zurück."""
    masked = introduce_missing_values_mcar(original, list(original.columns), missing_percent, seed=seed)
    imputed = imputer_fn(masked)
    return masked, imputed

def eval_imputation(original: pd.DataFrame, masked: pd.DataFrame, imputed: pd.DataFrame) -> pd.DataFrame:
    """RMSE/MAE auf den maskierten Positionen + Wilcoxon (paired) auf beobachteten Samples."""
    rows = []
    for col in original.columns:
        mask_positions = masked[col].isna() & original[col].notna()
        y_true = original.loc[mask_positions, col]
        y_hat = imputed.loc[mask_positions, col]
        if len(y_true) == 0:
            continue
        rmse = mean_squared_error(y_true, y_hat, squared=False)
        mae = mean_absolute_error(y_true, y_hat)
        # Distribution: Wilcoxon auf gesamten (nicht-NaN) Vektor (paired)
        o = original[col].dropna()
        p = imputed.loc[o.index, col].dropna()
        common = o.index.intersection(p.index)
        w_stat, w_p = wilcoxon(o.loc[common], p.loc[common])
        rows.append({"feature": col, "rmse": rmse, "mae": mae, "wilcoxon_stat": w_stat, "wilcoxon_p": w_p})
    return pd.DataFrame(rows)

def run_imputation_benchmark(df: pd.DataFrame, levels=MISSING_LEVELS) -> pd.DataFrame:
    all_rows = []
    for lvl in levels:
        for name, fn in IMPUTERS.items():
            masked, imputed = remove_and_reconstruct(df, fn, lvl, seed=RANDOM_SEED)
            res = eval_imputation(df, masked, imputed)
            res.insert(0, "missing_level", lvl)
            res.insert(1, "method", name)
            all_rows.append(res)
    return pd.concat(all_rows, ignore_index=True)

# Run (auf einer Datei) – für alle Dateien später iterieren
imp_results = run_imputation_benchmark(df0)
imp_results.head()


In [ ]:
# [Imputation Output] Speichern + einfache Aggregation für Paper-Tabellen
imp_results.to_csv(RESULTS_DIR / "imputation_results_single_file.csv", index=False)

summary = (imp_results
           .groupby(["missing_level", "method", "feature"], as_index=False)
           .agg(rmse=("rmse","mean"), mae=("mae","mean"), wilcoxon_p=("wilcoxon_p","mean")))
summary.head()


## 4) Outlier-Benchmark
**Was diese Zellen tun:**
- injizieren Outlier
- markieren Outlier als NaN (Z-Score, MAD, Isolation Forest)
- evaluieren via Varianz-Änderung und KS-Test


In [ ]:
# [Outliers] Detection / Handling

def detect_outliers_to_nan(df: pd.DataFrame, method: str, z_thresh: float = 3.0, mad_thresh: float = 3.0, contamination: float = 0.05) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        x = out[col].to_numpy()
        if method == "zscore":
            z = np.abs(zscore(pd.Series(x).fillna(np.nanmedian(x))))
            out[col] = np.where(z > z_thresh, np.nan, x)
        elif method == "mad":
            med = np.nanmedian(x)
            mad = median_abs_deviation(pd.Series(x).dropna(), nan_policy="omit")
            if mad == 0 or np.isnan(mad):
                continue
            out[col] = np.where(np.abs(x - med) > mad_thresh * mad, np.nan, x)
        elif method == "iforest":
            mask = ~np.isnan(x)
            if mask.sum() < 10:
                continue
            vals = x[mask].reshape(-1, 1)
            iso = IsolationForest(contamination=contamination, random_state=RANDOM_SEED)
            pred = iso.fit_predict(vals)
            x2 = x.copy()
            x2[mask] = np.where(pred == -1, np.nan, vals.flatten())
            out[col] = x2
        else:
            raise ValueError(f"Unknown method: {method}")
    return out

def eval_outlier_handling(original: pd.DataFrame, cleaned: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for col in original.columns:
        o = original[col].dropna()
        c = cleaned[col].dropna()
        if len(o) < 2 or len(c) < 2:
            continue
        o_var = np.var(o)
        c_var = np.var(c)
        var_red = np.nan
        if o_var != 0:
            var_red = (o_var - c_var) / o_var * 100.0
        ks_stat, ks_p = ks_2samp(o, c)
        rows.append({"feature": col, "variance_reduction_pct": var_red, "ks_stat": ks_stat, "ks_p": ks_p})
    return pd.DataFrame(rows)

OUTLIER_METHODS = ["zscore", "mad", "iforest"]

def run_outlier_benchmark(df: pd.DataFrame, levels=OUTLIER_LEVELS) -> pd.DataFrame:
    all_rows=[]
    for lvl in levels:
        corrupted = introduce_outliers_additive(df, list(df.columns), lvl, seed=RANDOM_SEED)
        for m in OUTLIER_METHODS:
            cleaned = detect_outliers_to_nan(corrupted, m)
            res = eval_outlier_handling(df, cleaned)
            res.insert(0, "outlier_level", lvl)
            res.insert(1, "method", m)
            all_rows.append(res)
    return pd.concat(all_rows, ignore_index=True)

out_results = run_outlier_benchmark(df0)
out_results.head()


In [ ]:
# [Outlier Output] Speichern
out_results.to_csv(RESULTS_DIR / "outlier_results_single_file.csv", index=False)

(out_results.groupby(["outlier_level","method","feature"], as_index=False)
 .agg(variance_reduction_pct=("variance_reduction_pct","mean"),
      ks_stat=("ks_stat","mean"),
      ks_p=("ks_p","mean"))
 .head())


## 5) Normalisierung (nach Imputation + Outlier-Handling)
**Was diese Zellen tun:**
- baut Kombinationen (MV → Outlier → Scaling)
- evaluiert Varianz/Skewness/Kurtosis + KS (deskriptiv)

**Hinweis:** KS ist nicht skaleninvariant; bei Normalisierung meist „signifikant“ – im Paper korrekt als deskriptiv behandelt.


In [ ]:
# [Normalization] Pipeline + Evaluation

SCALERS = {
    "minmax": MinMaxScaler(),
    "zscore": StandardScaler(),
    "robust": RobustScaler(),
}

def apply_scaler(df: pd.DataFrame, scaler) -> pd.DataFrame:
    arr = scaler.fit_transform(df.to_numpy())
    return pd.DataFrame(arr, columns=df.columns, index=df.index)

def eval_normalization(pre: pd.DataFrame, post: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for col in pre.columns:
        a = pre[col].dropna()
        b = post[col].dropna()
        common = a.index.intersection(b.index)
        if len(common) < 5:
            continue
        a = a.loc[common]; b = b.loc[common]
        a_var = np.var(a); b_var = np.var(b)
        var_red = np.nan if a_var == 0 else (a_var - b_var) / a_var * 100.0
        sk_red = abs(skew(a)) - abs(skew(b))
        ku_red = abs(kurtosis(a)) - abs(kurtosis(b))
        ks_stat, ks_p = ks_2samp(a, b)
        rows.append({"feature": col, "variance_reduction_pct": var_red, "skewness_reduction": sk_red,
                     "kurtosis_reduction": ku_red, "ks_stat": ks_stat, "ks_p": ks_p})
    return pd.DataFrame(rows)

def run_normalization_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    all_rows=[]
    # Fokus auf die beiden in deinem Paper diskutierten Imputer
    mv_candidates = {
        "knn": lambda d: knn_imputation(d, k=5),
        "locf": lambda d: locf_imputation(d),
    }
    for mv_name, mv_fn in mv_candidates.items():
        mv_done = mv_fn(df)
        for out_m in ["mad","iforest"]:
            out_done = detect_outliers_to_nan(mv_done, "mad" if out_m=="mad" else "iforest")
            # optional: outliers wieder imputieren (in deinem Paper: outlier->NaN; danach scaling)
            out_imputed = locf_imputation(out_done)  # simple fill for scaling stability
            for sc_name, sc in SCALERS.items():
                post = apply_scaler(out_imputed, sc)
                res = eval_normalization(out_imputed, post)
                res.insert(0, "mv_method", mv_name)
                res.insert(1, "outlier_method", out_m)
                res.insert(2, "scaler", sc_name)
                all_rows.append(res)
    return pd.concat(all_rows, ignore_index=True)

norm_results = run_normalization_benchmark(df0)
norm_results.head()


In [ ]:
# [Normalization Output] Speichern
norm_results.to_csv(RESULTS_DIR / "normalization_results_single_file.csv", index=False)
norm_results.groupby(["mv_method","outlier_method","scaler","feature"], as_index=False).mean().head()


## 6) Paper-Figuren (minimal)



In [ ]:
# [Figure] Beispiel: RMSE pro Methode (Gaze X)
sub = imp_results[(imp_results["feature"]=="Gaze X")]
pivot = sub.groupby(["missing_level","method"], as_index=False)["rmse"].mean()

plt.figure(figsize=(8,4))
for m in pivot["method"].unique():
    d = pivot[pivot["method"]==m]
    plt.plot(d["missing_level"], d["rmse"], marker="o", label=m)
plt.xlabel("Missingness (%)")
plt.ylabel("RMSE (Gaze X)")
plt.legend()
plt.tight_layout()
plt.show()
